# Analysis Notebook for all 3 Machine Learning Models

The three models suggested by our previous EDA involves:

1. Logistic Regression
2. Random Forest Classifer
3. XGBoost (gradient boosting)



Imbalanced data in cleaned_gas_monitoring.csv means we would need to ensure our data is balanced before we fit it into our models.


The steps we will be doing is 
1. Encoding and Normalising, 
2. Training BASELINE models, 
3. Applying SMOTE (Suggested) to balance classes, 
4. RandomisedSearchCV (with 5-fold cross validation) to search for optimal Hyperparameters 

In [ ]:
%load_ext autoreload
%autoreload 2
# As we are importing the same data in both notebooks, this command lets us link variables and functions across notebooks.

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Importing Necessary files

In [ ]:
from src.ML_cleaning_data import *
from src.FeatureEngineering import *

In [ ]:
# Loading data
df = load_cleaned_data()
df

,Time of Day,Temperature,Humidity,CO2_InfraredSensor,CO2_ElectroChemicalSensor,MetalOxideSensor_Unit1,MetalOxideSensor_Unit2,MetalOxideSensor_Unit3,MetalOxideSensor_Unit4,CO_GasSensor,Session ID,HVAC Operation Mode,Ambient Light Level,Activity Level
0,morning,19.63,53.83,125.486389,571.089440,478.554958,735.850412,654.625253,565.051969,2.0,7241,off,very_dim,Low Activity
1,morning,20.21,53.69,126.343018,575.789501,491.955151,740.282738,655.734327,557.078486,1.0,7241,ventilation_only,bright,Low Activity
2,morning,19.62,54.25,126.560695,585.543111,505.560808,737.112906,649.962421,558.065196,1.0,7241,eco_mode,very_bright,Low Activity
3,morning,20.10,50.48,113.504877,597.449961,515.142272,744.020651,676.150075,600.222210,1.0,7241,eco_mode,very_bright,High Activity
4,morning,19.89,48.42,92.766225,613.654875,535.664558,770.265441,720.274019,625.731124,1.0,7241,heating_active,moderate,Low Activity
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9730,evening,23.45,51.57,137.854649,544.124855,510.926349,726.784531,680.916215,620.292682,1.0,2586,maintenance_mode,very_bright,Low Activity
9731,evening,20.21,46.98,131.945968,541.756022,517.719693,738.901193,689.383365,625.100827,2.0,2586,ventilation_only,very_dim,Low Activity
9732,evening,23.71,49.16,136.422868,542.072190,512.607291,732.456099,683.197988,622.035384,2.0,2586,eco_mode,very_bright,Low Activity
9733,night,20.58,51.57,126.734430,561.716292,435.638480,707.447312,648.634308,581.583550,1.0,4761,cooling_active,very_bright,Low Activity


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9735 entries, 0 to 9734
Data columns (total 14 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Time of Day                9735 non-null   str    
 1   Temperature                9735 non-null   float64
 2   Humidity                   9735 non-null   float64
 3   CO2_InfraredSensor         9735 non-null   float64
 4   CO2_ElectroChemicalSensor  9735 non-null   float64
 5   MetalOxideSensor_Unit1     9735 non-null   float64
 6   MetalOxideSensor_Unit2     9735 non-null   float64
 7   MetalOxideSensor_Unit3     9735 non-null   float64
 8   MetalOxideSensor_Unit4     9735 non-null   float64
 9   CO_GasSensor               9735 non-null   float64
 10  Session ID                 9735 non-null   int64  
 11  HVAC Operation Mode        9735 non-null   str    
 12  Ambient Light Level        9735 non-null   str    
 13  Activity Level             9735 non-null   str    
dtypes: 

Data has been cleaned, moving on to identifying numerical and categorical columns

In [7]:
df.describe()

,Temperature,Humidity,CO2_InfraredSensor,CO2_ElectroChemicalSensor,MetalOxideSensor_Unit1,MetalOxideSensor_Unit2,MetalOxideSensor_Unit3,MetalOxideSensor_Unit4,CO_GasSensor,Session ID
count,9735.000000,9735.000000,9735.000000,9735.000000,9735.000000,9735.000000,9735.000000,9735.000000,9735.000000,9735.000000
mean,20.575252,51.038289,111.248229,578.624225,471.025216,728.175004,680.487947,612.264375,1.263790,5427.304674
std,2.557336,3.477224,31.584844,22.587891,51.454466,26.979628,56.361618,43.166075,0.749398,2592.874668
min,10.007079,39.690000,0.027967,408.599386,286.825662,611.168916,456.673900,412.791904,0.000000,1374.000000
25%,18.970000,49.200000,97.758507,559.336038,440.268421,712.917185,658.474815,585.367474,1.000000,3074.000000
50%,20.210000,51.570000,113.064142,579.351176,469.244105,726.784531,679.847421,609.514533,1.000000,5214.000000
75%,22.330000,52.955000,124.563132,595.578474,494.765605,741.259809,703.680497,636.161382,2.000000,7395.000000
max,34.301271,59.730000,237.873938,637.895790,632.891693,807.732943,906.213097,736.112361,4.000000,9658.000000


## Pre - Feature Engineering

We will only split, scale, and map simple text-to-number encoding for this round to generate BASELINE models.


When the baseline models are trained and evaluated, further enhancements to the models will be compared.

In the Data, we need to map the String values to numerical values for it to be readable to the model

In [9]:
activity_map = {'Low Activity': 0, 'Moderate Activity': 1, 'High Activity': 2}
df['Target_Activity'] = df['Activity Level'].map(activity_map)

In [10]:
light_map = {'very_dim': 0, 'dim': 1, 'moderate': 2, 'bright': 3, 'very_bright': 4}
df['Ambient_Light_Encoded'] = df['Ambient Light Level'].map(light_map).fillna(2)

In [11]:
df = pd.get_dummies(df, columns=['HVAC Operation Mode', 'Time of Day'], drop_first=True)

## Splitting and Scaling

In [12]:
# Removing all unnecessary columns for training
X = df.drop(columns=['Activity Level', 'Ambient Light Level', 'Target_Activity', 'Session ID'])
y = df['Target_Activity']

In [18]:
X_train, X_test, y_train, y_test = split_data(X, y)
X_train_scaled, X_test_scaled = scale_features(X_train, X_test)

In [20]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((7788, 18), (1947, 18), (7788,), (1947,))

In [35]:
%load_ext autoreload
%autoreload 2

evaluate_baseline_models(X_train_scaled, X_test_scaled, y_train, y_test)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Training Logistic Regression:

Logistic Regression Report:
                   precision    recall  f1-score   support

     Low Activity       0.65      0.87      0.74      1122
Moderate Activity       0.56      0.40      0.46       612
    High Activity       0.00      0.00      0.00       213

         accuracy                           0.63      1947
        macro avg       0.40      0.42      0.40      1947
     weighted avg       0.55      0.63      0.57      1947

Macro F1-Score (main evaluator): 0.4014

Training Random Forest:


/usr/local/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/usr/local/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/usr/local/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])



Random Forest Report:
                   precision    recall  f1-score   support

     Low Activity       0.78      0.81      0.80      1122
Moderate Activity       0.56      0.66      0.61       612
    High Activity       0.66      0.15      0.25       213

         accuracy                           0.69      1947
        macro avg       0.67      0.54      0.55      1947
     weighted avg       0.70      0.69      0.68      1947

Macro F1-Score (main evaluator): 0.5511

Training XGBoost:

XGBoost Report:
                   precision    recall  f1-score   support

     Low Activity       0.76      0.81      0.78      1122
Moderate Activity       0.54      0.59      0.57       612
    High Activity       0.32      0.15      0.21       213

         accuracy                           0.67      1947
        macro avg       0.54      0.52      0.52      1947
     weighted avg       0.65      0.67      0.65      1947

Macro F1-Score (main evaluator): 0.5188



EXPLORATION COMPLETE
The b

^Random forest shown to be the best performing baseline model compared to Logistic Regression and XGBoost.